In [1]:
import pandas as pd
import numpy as np
import warnings

In [2]:
df = pd.read_pickle(r"C:\Users\Almog\Desktop\Data Science\Projects\2026\Customer Personality Analysis\Pickle files\EDA_MC1.pkl")
df.head()

,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,...,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response,tenure_years
0,1957,Graduation,Single,58138.0,0,0,58,635,88,546,...,4,7,0,0,0,0,0,0,1,1.815
1,1954,Graduation,Single,46344.0,1,1,38,11,1,6,...,2,5,0,0,0,0,0,0,0,0.309
2,1965,Graduation,Together,71613.0,0,0,26,426,49,127,...,10,4,0,0,0,0,0,0,0,0.854
3,1984,Graduation,Together,26646.0,1,0,26,11,4,20,...,4,6,0,0,0,0,0,0,0,0.381
4,1981,PhD,Married,58293.0,1,0,94,173,43,118,...,6,5,0,0,0,0,0,0,0,0.441


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Year_Birth           2240 non-null   int64  
 1   Education            2240 non-null   string 
 2   Marital_Status       2240 non-null   string 
 3   Income               2240 non-null   float64
 4   Kidhome              2240 non-null   int64  
 5   Teenhome             2240 non-null   int64  
 6   Recency              2240 non-null   int64  
 7   MntWines             2240 non-null   int64  
 8   MntFruits            2240 non-null   int64  
 9   MntMeatProducts      2240 non-null   int64  
 10  MntFishProducts      2240 non-null   int64  
 11  MntSweetProducts     2240 non-null   int64  
 12  MntGoldProds         2240 non-null   int64  
 13  NumDealsPurchases    2240 non-null   int64  
 14  NumWebPurchases      2240 non-null   int64  
 15  NumCatalogPurchases  2240 non-null   i

#### Feature Engineering

Encoding Categorial Features with 'string' data type:

In [6]:
df['Marital_Status'].value_counts()

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: Int64

In [7]:
education_order = {
    "Basic": 0,
    "2n Cycle": 1,
    "Graduation": 2,
    "Master": 3,
    "PhD": 4
}

df["Education_ord"] = df["Education"].map(education_order)
df[["Education", "Education_ord"]].drop_duplicates().sort_values("Education_ord")

# drop original column
df.drop(columns=["Education"], inplace=True)

In [8]:
df["Marital_Status_clean"] = df["Marital_Status"].replace(
    ["Alone", "Absurd", "YOLO"],
    "Other"
)

marital_mapping = {
    "Married": 0,
    "Together": 1,
    "Single": 2,
    "Divorced": 3,
    "Widow": 4,
    "Other": 5
}

df["Marital_Status_encoded"] = df["Marital_Status_clean"].map(marital_mapping)
df.drop(columns=["Marital_Status", "Marital_Status_clean"], inplace=True)


In [9]:
df.head()

,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,...,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response,tenure_years,Education_ord,Marital_Status_encoded
0,1957,58138.0,0,0,58,635,88,546,172,88,...,0,0,0,0,0,0,1,1.815,2,2
1,1954,46344.0,1,1,38,11,1,6,2,1,...,0,0,0,0,0,0,0,0.309,2,2
2,1965,71613.0,0,0,26,426,49,127,111,21,...,0,0,0,0,0,0,0,0.854,2,1
3,1984,26646.0,1,0,26,11,4,20,10,3,...,0,0,0,0,0,0,0,0.381,2,1
4,1981,58293.0,1,0,94,173,43,118,46,27,...,0,0,0,0,0,0,0,0.441,4,0


In [10]:
df[['MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
                 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth']].head()

,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth
0,88,3,8,10,4,7
1,6,2,1,1,2,5
2,42,1,8,2,10,4
3,5,2,2,0,4,6
4,15,5,5,3,6,5


#### Creating New Features

| Feature Name               | Definition / Logic                      | What It Captures                        | Notes / Assumptions                                      |
| -------------------------- | --------------------------------------- | --------------------------------------- | -------------------------------------------------------- |
| `total_spending`           | Sum of all 2-year spending features     | Overall customer monetary value         | Foundational aggregate used by multiple derived features |
| `total_purchases`          | Sum of web + store purchases            | Purchase frequency and activity level   | Enables ratios and dominance indicators                  |
| `is_web_dominant`          | NumWebPurchases / total_purchases > 0.5 | Primary purchase channel preference     | Binary dominance simplifies interpretation               |
| `value_optimizer`          | High total_spending AND high deal_ratio | High spenders who rely on discounts     | Composite behavioral narrative feature                   |
| `spending_to_income_ratio` | total_spending / Income                 | Spending intensity relative to capacity | Requires handling zero or missing income                 |
| `family_load`              | Kidhome + Teenhome (binned)             | Household responsibility level          | Coarse bins preferred over exact counts                  |
| `is_long_tenure`           | tenure_years ≥ dataset median           | Relationship maturity with brand        | Tenure treated as a relative measure within the dataset rather than an absolute business threshold                                             |


In [13]:
# total_spendings
spending_cols = df[['MntWines','MntFruits',
                   'MntMeatProducts','MntFishProducts',
                   'MntSweetProducts','MntGoldProds']]
df['total_spending'] =  spending_cols.sum(axis = 1)

#total_purchases
purchase_cols = df[['NumStorePurchases','NumWebPurchases']]
df['total_purchases'] = purchase_cols.sum(axis = 1 )

#is_web_dominant
df['is_web_dominant'] = ((df['total_purchases'] > 0) & 
                         (df['NumWebPurchases'] / df['total_purchases'] > 0.5)).astype(int)

# value_optimizer
df['value_optimizer'] = (
    (df['total_spending'] > df['total_spending'].median()) &
    (
        df['NumDealsPurchases'] /
        df['total_purchases'].replace(0, np.nan)
        > 0.6
    )
).astype(int)

#spending_to_income_ratio
df['spending_to_income_ratio'] = df['total_spending']/df['Income']

#family_load
df['total_dependents'] = df['Kidhome'] + df['Teenhome']

df['family_load'] = pd.cut(
    df['total_dependents'],
    bins=[-1, 0, 2, np.inf],
    labels=[0, 1, 2]
).astype(int)

df.drop(columns='total_dependents', inplace=True)

#is_long_tenure
df['is_long_tenure'] = (
    df['tenure_years'] > df['tenure_years'].median()
).astype(int)

df.head()

,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,...,tenure_years,Education_ord,Marital_Status_encoded,total_spending,total_purchases,is_web_dominant,value_optimizer,spending_to_income_ratio,family_load,is_long_tenure
0,1957,58138.0,0,0,58,635,88,546,172,88,...,1.815,2,2,1617,12,1,0,0.027813,0,1
1,1954,46344.0,1,1,38,11,1,6,2,1,...,0.309,2,2,27,3,0,0,0.000583,1,0
2,1965,71613.0,0,0,26,426,49,127,111,21,...,0.854,2,1,776,18,0,0,0.010836,0,0
3,1984,26646.0,1,0,26,11,4,20,10,3,...,0.381,2,1,53,6,0,0,0.001989,1,0
4,1981,58293.0,1,0,94,173,43,118,46,27,...,0.441,4,0,422,11,0,0,0.007239,1,0


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Year_Birth                2240 non-null   int64  
 1   Income                    2240 non-null   float64
 2   Kidhome                   2240 non-null   int64  
 3   Teenhome                  2240 non-null   int64  
 4   Recency                   2240 non-null   int64  
 5   MntWines                  2240 non-null   int64  
 6   MntFruits                 2240 non-null   int64  
 7   MntMeatProducts           2240 non-null   int64  
 8   MntFishProducts           2240 non-null   int64  
 9   MntSweetProducts          2240 non-null   int64  
 10  MntGoldProds              2240 non-null   int64  
 11  NumDealsPurchases         2240 non-null   int64  
 12  NumWebPurchases           2240 non-null   int64  
 13  NumCatalogPurchases       2240 non-null   int64  
 14  NumStore

In [15]:
df.to_pickle(r"C:\Users\Almog\Desktop\Data Science\Projects\2026\Customer Personality Analysis\Pickle files\FE_MC1.pkl")